In [ ]:
# ==================== 自動處理「TRAIN 0-5」資料夾 ====================

from pathlib import Path
import pandas as pd

# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. 設定您的資料夾路徑（請確認路徑正確）
TRAIN_FOLDER = "/content/drive/MyDrive//Time-LLM-main/TRAIN 0-5"   # ← 如果資料夾名稱不同，請修改這裡

# 3. 自動掃描資料夾內所有 CSV 並合併成 train.csv
csv_files = list(Path(TRAIN_FOLDER).glob("*.csv"))

print(f"找到 {len(csv_files)} 個 CSV 檔案（資料夾：TRAIN 0-5）")

df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    # 只保留需要的兩欄（您可以自行增加其他欄位）
    if 'disp_x_diff' in df.columns and 'disp_z_diff' in df.columns:
        df = df[['disp_x_diff', 'disp_z_diff']]
        df_list.append(df)
    else:
        print(f"警告：檔案 {file.name} 缺少 disp_x_diff 或 disp_z_diff 欄位，已跳過")

# 合併並儲存
train_df = pd.concat(df_list, ignore_index=True)
train_df.to_csv("/content/drive/MyDrive/train.csv", index=False)

print(f"✅ 合併完成！共 {len(train_df)} 筆資料，已儲存為 train.csv")
print(f"train.csv 路徑：/content/drive/MyDrive/train.csv")

Mounted at /content/drive
找到 43 個 CSV 檔案（資料夾：TRAIN 0-5）
✅ 合併完成！共 28462 筆資料，已儲存為 train.csv
train.csv 路徑：/content/drive/MyDrive/train.csv


In [ ]:
from huggingface_hub import login

# 執行這段後，下方會出現一個對話框，請貼上你的 Hugging Face Access Token
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate bitsandbytes

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive
# ==========================================
# 1. 重建 ACE 框架
# ==========================================
class ACE_Playbook:
    def __init__(self):
        self.playbook = []
    def generator(self, trend, top5_lags, expert_knowledge):
        return f"[ACE 動態上下文]\n趨勢：{trend}\n前五大滯後：{top5_lags}\n專家知識：{expert_knowledge}\n建議：溫度與壓力對位移影響大。"
    def curator(self, new_strategy):
        self.playbook.append(new_strategy)
        return self.playbook[-3:]

# ==========================================
# 2. 重建 資料集類別 (解決您的 NameError)
# ==========================================
class LatheDataset(Dataset):
    def __init__(self, csv_path, train_mean=None, train_std=None):
        self.df = pd.read_csv(csv_path).dropna()
        raw_data = torch.tensor(self.df[['disp_x_diff', 'disp_z_diff']].values, dtype=torch.float32)

        if train_mean is None or train_std is None:
            self.mean = raw_data.mean(dim=0)
            self.std = raw_data.std(dim=0)
        else:
            self.mean = train_mean
            self.std = train_std

        self.data = (raw_data - self.mean) / (self.std + 1e-8)
        self.seq_len = 16

    def __len__(self):
        return len(self.data) - self.seq_len + 1

    def __getitem__(self, idx):
        return self.data[idx : idx + self.seq_len]

# ==========================================
# 3. 重建 核心模型類別
# ==========================================
class TimeLLMWithACE(nn.Module):
    def __init__(self, V_prime=200):
        super().__init__()
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        self.llm = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-2-7b-hf",
            dtype=torch.float16,
            device_map="auto",
            quantization_config=bnb_config
        )
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
        self.llm.eval()
        for param in self.llm.parameters():
            param.requires_grad = False

        self.patch_embedding = nn.Linear(2, V_prime)
        self.W = nn.Parameter(torch.empty(V_prime, self.llm.config.vocab_size))
        nn.init.xavier_uniform_(self.W)
        self.reprogram_norm = nn.LayerNorm(self.llm.config.hidden_size)
        self.output_projection = nn.Linear(self.llm.config.hidden_size, 2)
        self.ace = ACE_Playbook()

    def forward(self, patches, trend, top5_lags, expert_knowledge):
        new_strategy = self.ace.generator(trend, top5_lags, expert_knowledge)
        prompt = f"[ACE 動態上下文]\n{new_strategy}\n請預測下一個時間步的位移變化。"
        batch_size = patches.shape[0]

        patch_emb = self.patch_embedding(patches)
        llm_weight_fp32 = self.llm.get_input_embeddings().weight.to(torch.float32)
        E_prime = torch.matmul(self.W, llm_weight_fp32)
        reprogrammed = torch.matmul(patch_emb, E_prime)
        reprogrammed = self.reprogram_norm(reprogrammed).to(torch.float16)

        inputs = self.tokenizer(prompt, return_tensors="pt").to(patch_emb.device)
        text_emb = self.llm.get_input_embeddings()(inputs.input_ids)
        text_emb = text_emb.expand(batch_size, -1, -1)
        inputs_embeds = torch.cat([reprogrammed, text_emb], dim=1)

        outputs = self.llm(inputs_embeds=inputs_embeds, output_hidden_states=True)
        hidden = outputs.hidden_states[-1].mean(dim=1)
        return self.output_projection(hidden.to(torch.float32))

print("✅ 所有地基類別（Dataset, Model, ACE）皆已重新載入記憶體！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.5 MB/s eta 0:00:00
✅ 所有地基類別（Dataset, Model, ACE）皆已重新載入記憶體！


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import re
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==========================================
# 1. 智慧型資料對齊與合併模組
# ==========================================
def prepare_training_data(env_path, train_folder, save_path):
    print("開始進行智慧型資料對齊與合併...")

    # 讀取並建立環境字典
    env_df = pd.read_excel(env_path)
    new_columns = ['日期', '段1_轉速', '段1_進給', '段1_時間', '段2_轉速', '段2_進給', '段2_時間', '段3_轉速', '段3_進給', '段3_時間', '控溫模式', '溫度']
    env_df.columns = new_columns
    env_df = env_df.drop(0).reset_index(drop=True)

    env_dict = {}
    for _, row in env_df.iterrows():
        if pd.isna(row['日期']): continue

        date_str = str(int(row['日期']))
        temp_mode = str(row['控溫模式'])
        temp_val = str(row['溫度'])
        prompt = f"機台環境為{temp_mode}，設定溫度 {temp_val} 度。"

        stages = []
        if pd.notna(row['段1_轉速']): stages.append(f"第一段轉速 {int(row['段1_轉速'])}rpm，進給 {int(row['段1_進給'])}")
        if pd.notna(row['段2_轉速']): stages.append(f"第二段轉速 {int(row['段2_轉速'])}rpm，進給 {int(row['段2_進給'])}")
        if stages:
            prompt += " 加工參數：" + "；".join(stages) + "。"

        env_dict[date_str] = prompt

    print(f"成功從 Excel 建立環境字典，共 {len(env_dict)} 種實驗設定。")

    # 掃描並注入提示詞
    csv_files = list(Path(train_folder).glob("*.csv"))
    print(f"找到 {len(csv_files)} 個 CSV 檔案，準備解析檔名並注入提示詞...")

    df_list = []
    for file in csv_files:
        match = re.search(r'2020\d{4}', file.name)
        if not match:
            print(f"警告：無法從檔名 {file.name} 找到日期，跳過。")
            continue

        date_key = match.group(0)
        matched_prompt = env_dict.get(date_key, "機台運作中，無特殊環境紀錄。")

        df = pd.read_csv(file)
        if 'disp_x_diff' in df.columns and 'disp_z_diff' in df.columns:
            df = df[['disp_x_diff', 'disp_z_diff']].copy()
            df['expert_prompt'] = matched_prompt
            df_list.append(df)

    # 合併並儲存
    train_env_df = pd.concat(df_list, ignore_index=True)
    train_env_df.to_csv(save_path, index=False)

    print("-" * 50)
    print(f"智慧合併完成，共 {len(train_env_df)} 筆資料。")
    print(f"新的訓練集已儲存至：{save_path}")
    print("-" * 50)
    return save_path

# ==========================================
# 2. ACE 框架與核心模型
# ==========================================
class ACE_Playbook:
    def __init__(self):
        self.playbook = []

    def generator(self, trend, top5_lags, expert_knowledge):
        return f"[ACE 動態上下文]\n趨勢：{trend}\n前五大滯後：{top5_lags}\n專家知識：{expert_knowledge}\n建議：溫度與壓力對位移影響大。"

    def curator(self, new_strategy):
        self.playbook.append(new_strategy)
        return self.playbook[-3:]

class TimeLLMWithACE(nn.Module):
    def __init__(self, V_prime=200):
        super().__init__()
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        self.llm = AutoModelForCausalLM.from_pretrained(
            "meta-llama/Llama-2-7b-hf",
            dtype=torch.float16,
            device_map="auto",
            quantization_config=bnb_config
        )
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
        self.llm.eval()

        for param in self.llm.parameters():
            param.requires_grad = False

        self.patch_embedding = nn.Linear(2, V_prime)
        self.W = nn.Parameter(torch.empty(V_prime, self.llm.config.vocab_size))
        nn.init.xavier_uniform_(self.W)
        self.reprogram_norm = nn.LayerNorm(self.llm.config.hidden_size)
        self.output_projection = nn.Linear(self.llm.config.hidden_size, 2)
        self.ace = ACE_Playbook()

    def forward(self, patches, trend, top5_lags, expert_knowledge):
        new_strategy = self.ace.generator(trend, top5_lags, expert_knowledge)
        prompt = f"[ACE 動態上下文]\n{new_strategy}\n請預測下一個時間步的位移變化。"
        batch_size = patches.shape[0]

        patch_emb = self.patch_embedding(patches)
        llm_weight_fp32 = self.llm.get_input_embeddings().weight.to(torch.float32)
        E_prime = torch.matmul(self.W, llm_weight_fp32)
        reprogrammed = torch.matmul(patch_emb, E_prime)
        reprogrammed = self.reprogram_norm(reprogrammed).to(torch.float16)

        inputs = self.tokenizer(prompt, return_tensors="pt").to(patch_emb.device)
        text_emb = self.llm.get_input_embeddings()(inputs.input_ids)
        text_emb = text_emb.expand(batch_size, -1, -1)

        inputs_embeds = torch.cat([reprogrammed, text_emb], dim=1)
        outputs = self.llm(inputs_embeds=inputs_embeds, output_hidden_states=True)

        hidden = outputs.hidden_states[-1].mean(dim=1)
        return self.output_projection(hidden.to(torch.float32))

# ==========================================
# 3. 支援動態讀取提示詞的資料集
# ==========================================
class LatheDatasetWithPrompt(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path).dropna()
        self.raw_data = torch.tensor(self.df[['disp_x_diff', 'disp_z_diff']].values, dtype=torch.float32)

        self.mean = self.raw_data.mean(dim=0)
        self.std = self.raw_data.std(dim=0)
        self.data = (self.raw_data - self.mean) / (self.std + 1e-8)

        self.prompts = self.df['expert_prompt'].values.tolist()
        self.seq_len = 16

    def __len__(self):
        return len(self.data) - self.seq_len + 1

    def __getitem__(self, idx):
        seq = self.data[idx : idx + self.seq_len]
        prompt = self.prompts[idx + self.seq_len - 1]
        return seq, prompt

# ==========================================
# 4. 訓練迴圈
# ==========================================
def train_model_integrated(model, train_loader, epochs=20, lr=1e-4):
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.MSELoss()
    model.train()

    print(f"開始訓練程序，共 {epochs} Epochs")
    print("=" * 50)

    train_loss_history = []

    for epoch in range(epochs):
        total_loss = 0
        last_context_seen = ""

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for batch_seqs, batch_prompts in progress_bar:
            optimizer.zero_grad()

            inputs = batch_seqs.cuda()
            target = inputs[:, -1, :].cuda()

            current_env_text = batch_prompts[0]

            trend = "upward" if inputs.mean() > 0 else "downward"
            abs_x_fluctuations = torch.abs(inputs[0, :, 0])
            _, top_indices = torch.topk(abs_x_fluctuations, k=5)
            top5_lags = top_indices.tolist()

            pred = model(inputs, trend, top5_lags, expert_knowledge=current_env_text)
            last_context_seen = model.ace.generator(trend, top5_lags, current_env_text)

            loss = criterion(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        train_loss_history.append(avg_loss)

        input_data_x = inputs[0, :, 0].cpu().detach().numpy().tolist()
        predicted_output = pred[0].cpu().detach().numpy().tolist()
        actual_output = target[0].cpu().detach().numpy().tolist()

        print(f"\n[Epoch {epoch+1}/{epochs}] 結算：")
        print(f"Average MSE Loss : {avg_loss:.5f}")
        print("-" * 50)

        print("[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：")
        print([round(num, 4) for num in input_data_x])
        print(f"\n[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：\n{last_context_seen}")
        print(f"\n[模型輸出] 預測下一時間步位移差：X 軸 {predicted_output[0]:.4f}, Z 軸 {predicted_output[1]:.4f}")
        print(f"[真實答案] 實際下一時間步位移差：X 軸 {actual_output[0]:.4f}, Z 軸 {actual_output[1]:.4f}")
        print("=" * 50)

    save_path = "/content/drive/MyDrive/time_llm_ace_env_v2.pth"
    torch.save(model.state_dict(), save_path)
    print(f"訓練完成，模型權重已儲存至：{save_path}")
    return train_loss_history

# ==========================================
# 5. 主程式執行區塊
# ==========================================
if __name__ == "__main__":
    print("系統初始化中...")

    # 設定檔案路徑
    ENV_EXCEL_PATH = "/content/drive/MyDrive/Time-LLM-main/檔案環境設定總表.xlsx"
    TRAIN_FOLDER_PATH = "/content/drive/MyDrive/Time-LLM-main/TRAIN 0-5"
    GENERATED_CSV_PATH = "/content/drive/MyDrive/train_env.csv"

    try:
        # 步驟一：自動執行資料的清洗、對齊與合併
        final_csv_path = prepare_training_data(
            env_path=ENV_EXCEL_PATH,
            train_folder=TRAIN_FOLDER_PATH,
            save_path=GENERATED_CSV_PATH
        )

        # 步驟二：載入含有提示詞的新資料集
        print("載入訓練集資料...")
        train_dataset = LatheDatasetWithPrompt(final_csv_path)
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

        # 步驟三：初始化模型
        print("載入核心模型與權重...")
        model = TimeLLMWithACE().cuda()

        # 步驟四：開始訓練
        loss_history = train_model_integrated(
            model=model,
            train_loader=train_loader,
            epochs=20
        )

    except Exception as e:
        print(f"執行期間發生錯誤：{e}")

系統初始化中...
開始進行智慧型資料對齊與合併...
成功從 Excel 建立環境字典，共 56 種實驗設定。
找到 43 個 CSV 檔案，準備解析檔名並注入提示詞...
--------------------------------------------------
智慧合併完成，共 28462 筆資料。
新的訓練集已儲存至：/content/drive/MyDrive/train_env.csv
--------------------------------------------------
載入訓練集資料...
載入核心模型與權重...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...


[Epoch 16/20] 結算：
Average MSE Loss : 0.00903
--------------------------------------------------
[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：
[-0.067, -0.3546, -0.067, -0.067, -0.067, -0.3553, -0.067, 0.2213, -0.3553, -0.067, -0.3546, -0.067, -0.067, -0.067, -0.067, -0.067]

[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：
[ACE 動態上下文]
趨勢：upward
前五大滯後：[5, 8, 10, 1, 7]
專家知識：機台環境為恆溫，設定溫度 15 (parameter 8) 度。 加工參數：第一段轉速 2000rpm，進給 5000；第二段轉速 1000rpm，進給 5000。
建議：溫度與壓力對位移影響大。

[模型輸出] 預測下一時間步位移差：X 軸 -0.0739, Z 軸 0.1922
[真實答案] 實際下一時間步位移差：X 軸 -0.0670, Z 軸 0.1690



[Epoch 17/20] 結算：
Average MSE Loss : 0.00754
--------------------------------------------------
[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：
[-0.3553, 0.2213, -0.3553, -0.067, 0.2213, -0.3553, -0.067, -0.3546, 0.2205, -0.067, 0.2213, -0.3553, -0.067, -0.067, -0.067, 0.2213]

[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：
[ACE 動態上下文]
趨勢：downward
前五大滯後：[2, 0, 5, 11, 7]
專家知識：機台環境為變溫，設定溫度 20→30 (rise2°C per hour) 度。 加工參數：第一段轉速 1000rpm，進給 5000。
建議：溫度與壓力對位移影響大。

[模型輸出] 預測下一時間步位移差：X 軸 0.2166, Z 軸 0.1344
[真實答案] 實際下一時間步位移差：X 軸 0.2213, Z 軸 0.1690



[Epoch 18/20] 結算：
Average MSE Loss : 0.01547
--------------------------------------------------
[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：
[-0.3546, -0.067, -0.067, 0.2205, -0.067, 0.7972, -0.067, -0.3553, 0.5089, -0.3546, 0.5089, -0.067, -0.3553, -0.067, -0.067, -0.067]

[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：
[ACE 動態上下文]
趨勢：upward
前五大滯後：[5, 10, 8, 12, 7]
專家知識：機台環境為恆溫，設定溫度 30 (parameter 5) 度。 加工參數：第一段轉速 2000rpm，進給 5000；第二段轉速 0rpm，進給 0。
建議：溫度與壓力對位移影響大。

[模型輸出] 預測下一時間步位移差：X 軸 -0.0717, Z 軸 0.0559
[真實答案] 實際下一時間步位移差：X 軸 -0.0670, Z 軸 0.0078



[Epoch 19/20] 結算：
Average MSE Loss : 0.00808
--------------------------------------------------
[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：
[-0.067, -0.067, -0.067, -0.067, -0.067, -0.067, -0.067, -0.3553, 0.2213, -0.067, -0.067, -0.3553, -0.067, 0.2213, -0.3553, 0.2213]

[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：
[ACE 動態上下文]
趨勢：downward
前五大滯後：[14, 11, 7, 13, 8]
專家知識：機台環境為恆溫，設定溫度 30 (parameter 8) 度。 加工參數：第一段轉速 1800rpm，進給 5000；第二段轉速 0rpm，進給 0。
建議：溫度與壓力對位移影響大。

[模型輸出] 預測下一時間步位移差：X 軸 0.1369, Z 軸 -0.0629
[真實答案] 實際下一時間步位移差：X 軸 0.2213, Z 軸 0.0078



[Epoch 20/20] 結算：
Average MSE Loss : 0.00911
--------------------------------------------------
[輸入特徵 1 - 歷史數值] 過去 16 步 X 軸位移正規化數值：
[-0.067, 0.5089, -0.067, 0.2205, -0.067, 0.2213, -0.067, -0.067, -0.3553, -0.067, -0.067, -0.067, -0.067, -0.067, -0.3546, 0.2205]

[輸入特徵 2 - 文字語意] ACE 動態上下文提示詞：
[ACE 動態上下文]
趨勢：downward
前五大滯後：[1, 8, 14, 5, 3]
專家知識：機台環境為變溫，設定溫度 30→20→30 (diff -+2°C per 20min) 度。 加工參數：第一段轉速 1000rpm，進給 5000。
建議：溫度與壓力對位移影響大。

[模型輸出] 預測下一時間步位移差：X 軸 0.1369, Z 軸 0.0776
[真實答案] 實際下一時間步位移差：X 軸 0.2205, Z 軸 0.0078
訓練完成，模型權重已儲存至：/content/drive/MyDrive/time_llm_ace_env_v2.pth


In [ ]:
import pandas as pd
import torch
import numpy as np
import re
import os
from pathlib import Path

# ==========================================
# 競賽解答生成與檔案輸出腳本 (訓練/考卷 完美對齊版)
# ==========================================
def generate_competition_answers_perfect_match():
    print("🔍 系統初始化：準備批次處理並匯出預測檔案...")

    # ==========================================
    # 🎯 兩端欄位名稱已寫死，請勿修改！
    # ==========================================
    # 1. 訓練集 (train_env.csv) 裡面的欄位名稱 (當初合併的原始名稱)
    TRAIN_COL_X = 'disp_x_diff'
    TRAIN_COL_Z = 'disp_z_diff'

    # 2. 測驗集 (初賽測驗用數據) 官方考卷的名稱 (含大小寫與空格)
    TEST_COL_X = 'Disp. X'
    TEST_COL_Z = 'Disp. Z'

    # 路徑設定
    ENV_EXCEL_PATH = "/content/drive/MyDrive/Time-LLM-main/檔案環境設定總表.xlsx"
    TEST_FOLDER_PATH = "/content/drive/MyDrive/Time-LLM-main/初賽測驗用數據"
    TRAIN_CSV_PATH = "/content/drive/MyDrive/train_env.csv"
    MODEL_PATH = "/content/drive/MyDrive/time_llm_ace_env_v2.pth"

    # 建立輸出資料夾
    OUTPUT_FOLDER = "/content/drive/MyDrive/Time-LLM-main/預測結果輸出"
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    print(f"📁 預測結果將獨立儲存於：{OUTPUT_FOLDER}")

    # 1. 取得訓練集的正規化參數 (對齊 TRAIN_COL)
    try:
        train_df = pd.read_csv(TRAIN_CSV_PATH).dropna()
        train_raw = torch.tensor(train_df[[TRAIN_COL_X, TRAIN_COL_Z]].values, dtype=torch.float32)
        train_mean = train_raw.mean(dim=0)
        train_std = train_raw.std(dim=0)
    except KeyError as e:
        print(f"錯誤：在訓練集 (train_env.csv) 找不到欄位 {e}。")
        return

    # 2. 建立環境字典
    env_df = pd.read_excel(ENV_EXCEL_PATH)
    env_df.columns = ['日期', '段1_轉速', '段1_進給', '段1_時間', '段2_轉速', '段2_進給', '段2_時間', '段3_轉速', '段3_進給', '段3_時間', '控溫模式', '溫度']
    env_df = env_df.drop(0).reset_index(drop=True)

    env_dict = {}
    for _, row in env_df.iterrows():
        if pd.isna(row['日期']): continue
        date_str = str(int(row['日期']))
        prompt = f"機台環境為{str(row['控溫模式'])}，設定溫度 {str(row['溫度'])} 度。"
        stages = []
        if pd.notna(row['段1_轉速']): stages.append(f"第一段轉速 {int(row['段1_轉速'])}rpm，進給 {int(row['段1_進給'])}")
        if pd.notna(row['段2_轉速']): stages.append(f"第二段轉速 {int(row['段2_轉速'])}rpm，進給 {int(row['段2_進給'])}")
        if stages: prompt += " 加工參數：" + "；".join(stages) + "。"
        env_dict[date_str] = prompt

    # 3. 載入模型
    print(" 載入 Time-LLM + ACE 核心模型...")
    model = TimeLLMWithACE().cuda()
    model.load_state_dict(torch.load(MODEL_PATH))
    model.eval()

    print("-" * 60)
    print("-" * 60)

    test_files = list(Path(TEST_FOLDER_PATH).glob("*.csv"))
    total_predictions = 0

    with torch.no_grad():
        for file in test_files:
            match = re.search(r'2020\d{4}', file.name)
            date_key = match.group(0) if match else None
            matched_prompt = env_dict.get(date_key, "機台運作中，無特殊環境紀錄。")

            # 讀取測試檔 (對齊 TEST_COL)
            df = pd.read_csv(file)

            if TEST_COL_X not in df.columns or TEST_COL_Z not in df.columns:
                print(f"⚠️ {file.name} 缺少考題欄位 '{TEST_COL_X}' 或 '{TEST_COL_Z}'，已跳過。")
                continue

            df_filled = df.copy()
            df_filled[TEST_COL_X] = pd.to_numeric(df_filled[TEST_COL_X], errors='coerce')
            df_filled[TEST_COL_Z] = pd.to_numeric(df_filled[TEST_COL_Z], errors='coerce')

            # 尋找測試檔中的缺失值 (考題)
            missing_indices = df_filled[df_filled[TEST_COL_X].isna() | df_filled[TEST_COL_Z].isna()].index
            file_pred_count = 0

            for idx in missing_indices:
                # 滾動式擷取歷史資料
                if idx >= 16:
                    history = df_filled.loc[idx-16 : idx-1, [TEST_COL_X, TEST_COL_Z]].values
                else:
                    pad_length = 16 - idx
                    pad_array = np.tile([train_mean[0].item(), train_mean[1].item()], (pad_length, 1))
                    if idx > 0:
                        avail_history = df_filled.loc[0 : idx-1, [TEST_COL_X, TEST_COL_Z]].values
                        history = np.vstack((pad_array, avail_history))
                    else:
                        history = pad_array

                if np.isnan(history).any():
                    history[np.isnan(history[:, 0]), 0] = train_mean[0].item()
                    history[np.isnan(history[:, 1]), 1] = train_mean[1].item()

                # 張量處理與推論
                history_tensor = torch.tensor(history, dtype=torch.float32)
                norm_history = (history_tensor - train_mean) / (train_std + 1e-8)
                inputs = norm_history.unsqueeze(0).cuda()

                trend = "upward" if inputs.mean() > 0 else "downward"
                abs_x_fluctuations = torch.abs(inputs[0, :, 0])
                _, top_indices = torch.topk(abs_x_fluctuations, k=5)
                top5_lags = top_indices.tolist()

                pred = model(inputs, trend, top5_lags, expert_knowledge=matched_prompt)

                # 還原單位
                std_x, mean_x = train_std[0].item(), train_mean[0].item()
                std_z, mean_z = train_std[1].item(), train_mean[1].item()

                pred_x = pred[0][0].item() * std_x + mean_x
                pred_z = pred[0][1].item() * std_z + mean_z

                # 🎯 填回測驗集指定的考卷欄位中
                df_filled.at[idx, TEST_COL_X] = round(pred_x, 6)
                df_filled.at[idx, TEST_COL_Z] = round(pred_z, 6)

                file_pred_count += 1
                total_predictions += 1

            # 將結果存出
            output_filepath = os.path.join(OUTPUT_FOLDER, f"Answer_{file.name}")
            df_filled.to_csv(output_filepath, index=False)

            print(f" 完成處理：{file.name} | 成功填入 {file_pred_count} 筆數據")

    print("=" * 60)
    print(f"。總計填入了 {total_predictions} 筆數據。")


if __name__ == "__main__":
    generate_competition_answers_perfect_match()

🔍 系統初始化：準備批次處理並匯出預測檔案...
📁 預測結果將獨立儲存於：/content/drive/MyDrive/Time-LLM-main/預測結果輸出
🧠 載入 Time-LLM + ACE 核心模型...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

------------------------------------------------------------
🚀 開始去 13 個測驗檔中尋找 [Disp. X] 與 [Disp. Z] 的考題並填答...
------------------------------------------------------------
✅ 完成處理：_20200915_GV1-1203_2000rpm_XZ-5m-min_5H(wAC-from0-20to30C).csv | 成功填入 418 筆考題
✅ 完成處理：_20200917_GV1-1203_1000rpm_XZ-5m-min_5H(wAC-from0-15to25C).csv | 成功填入 418 筆考題
✅ 完成處理：_20201008_GV1-1203_1000rpm_XZ-5m-min_6H(wAC-from0-25to15to25C).csv | 成功填入 1539 筆考題
✅ 完成處理：_20200715_GV1-1203_1-8k_XZ-1-5m-min_2-5H+Stop1H+1-2k_XZ-5m-min_2-5H(wAC-from0-20Cto25C).csv | 成功填入 507 筆考題
✅ 完成處理：_20200720_GV1-1203_2k+1krpm_XZ-5m-min_6H(wAC-from0-25Cto20C).csv | 成功填入 507 筆考題
✅ 完成處理：_20200721_GV1-1203_1k+2krpm_XZ-5m-min_6H(wAC-from0-25Cto20C).csv | 成功填入 507 筆考題
✅ 完成處理：_20200819_GV1-1203_1-8k_XZ-1m-min_2-5H+Stop1H+1-2k_XZ-5m-min_2-5H(wAC-from0-25Cto20C).csv | 成功填入 504 筆考題
✅ 完成處理：_20200908_GV1-1203_2000rpm_XZ-5m-min_6H(wAC-from0-15to35C).csv | 成功填入 507 筆考題
✅ 完成處理：_20200909_GV1-1203_1000rpm_XZ-5m-min_6H(wAC-from0-35to15C).csv | 成功填入 507 筆考題

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import math

# ==========================================
# 替代基準 RMSE 結算模組 (Pseudo-RMSE)
# ==========================================
def calculate_pseudo_rmse():

    # 欄位設定
    TEST_COL_X = 'Disp. X'
    TEST_COL_Z = 'Disp. Z'

    # 路徑設定
    ORIGINAL_FOLDER = "/content/drive/MyDrive/Time-LLM-main/初賽測驗用數據"
    ANSWER_FOLDER = "/content/drive/MyDrive/Time-LLM-main/預測結果輸出"

    original_files = list(Path(ORIGINAL_FOLDER).glob("*.csv"))
    report_data = []

    for orig_file in original_files:
        ans_filepath = os.path.join(ANSWER_FOLDER, f"Answer_{orig_file.name}")

        # 確保預測檔案存在
        if not os.path.exists(ans_filepath):
            continue

        df_orig = pd.read_csv(orig_file)
        df_ans = pd.read_csv(ans_filepath)

        if TEST_COL_X not in df_orig.columns or TEST_COL_Z not in df_orig.columns:
            continue

        df_orig[TEST_COL_X] = pd.to_numeric(df_orig[TEST_COL_X], errors='coerce')
        df_orig[TEST_COL_Z] = pd.to_numeric(df_orig[TEST_COL_Z], errors='coerce')

        # 尋找原始檔案中確實為考題(缺失值)的索引位置
        missing_mask = df_orig[TEST_COL_X].isna() | df_orig[TEST_COL_Z].isna()
        missing_indices = df_orig[missing_mask].index

        if len(missing_indices) == 0:
            continue

        # 建立替代基準：使用 Pandas 的線性插值填補，模擬傳統統計學的預測結果
        df_pseudo = df_orig.copy()
        df_pseudo[TEST_COL_X] = df_pseudo[TEST_COL_X].interpolate(method='linear', limit_direction='both')
        df_pseudo[TEST_COL_Z] = df_pseudo[TEST_COL_Z].interpolate(method='linear', limit_direction='both')

        # 僅取出考題位置的數值進行嚴格比對
        pseudo_x = df_pseudo.loc[missing_indices, TEST_COL_X].values
        pseudo_z = df_pseudo.loc[missing_indices, TEST_COL_Z].values

        model_x = df_ans.loc[missing_indices, TEST_COL_X].values
        model_z = df_ans.loc[missing_indices, TEST_COL_Z].values

        # 計算模型與線性插值基準的 RMSE
        rmse_x = math.sqrt(np.mean((pseudo_x - model_x) ** 2))
        rmse_z = math.sqrt(np.mean((pseudo_z - model_z) ** 2))

        display_name = orig_file.name if len(orig_file.name) <= 45 else orig_file.name[:42] + "..."

        report_data.append({
            "檔案名稱": display_name,
            "預測題數": len(missing_indices),
            "X軸 相對RMSE": round(rmse_x, 5),
            "Z軸 相對RMSE": round(rmse_z, 5)
        })

    if not report_data:
        print("結算失敗，找不到對應的預測檔案或缺失值。")
        return

    # 印出最終報告總表
    print("\n" + "=" * 80)
    print("模型預測 vs 線性插值基準 (Pseudo-RMSE) 評估報告")
    print("=" * 80)
    print(f"{'檔案名稱':<45} | {'預測題數':<8} | {'X軸 相對RMSE':<12} | {'Z軸 相對RMSE':<12}")
    print("-" * 80)

    # 按照 X 軸誤差由小到大排序
    report_data = sorted(report_data, key=lambda x: x['X軸 相對RMSE'])

    for row in report_data:
        print(f"{row['檔案名稱']:<45} | {row['預測數量']:<10} | {row['X軸 相對RMSE']:<14.5f} | {row['Z軸 相對RMSE']:<14.5f}")

    print("=" * 80)

if __name__ == "__main__":
    calculate_pseudo_rmse()


模型預測 vs 線性插值基準 (Pseudo-RMSE) 評估報告
檔案名稱                                          | 預測題數     | X軸 相對RMSE    | Z軸 相對RMSE   
--------------------------------------------------------------------------------
_20200915_GV1-1203_2000rpm_XZ-5m-min_5H(wA... | 418        | 0.95519        | 2.87283       
_20201007_GV1-1203_1000rpm_XZ-5m-min_6H(wA... | 507        | 1.17440        | 8.81399       
_20200917_GV1-1203_1000rpm_XZ-5m-min_5H(wA... | 418        | 1.22094        | 4.33683       
_20201015_GV1-1203_1000rpm_XZ-5m-min_6H(wA... | 507        | 1.42055        | 2.13510       
_20200908_GV1-1203_2000rpm_XZ-5m-min_6H(wA... | 507        | 2.25389        | 0.10537       
_20200909_GV1-1203_1000rpm_XZ-5m-min_6H(wA... | 507        | 2.61725        | 1.61042       
_20201008_GV1-1203_1000rpm_XZ-5m-min_6H(wA... | 1539       | 3.17425        | 3.17151       
_20200720_GV1-1203_2k+1krpm_XZ-5m-min_6H(w... | 507        | 3.37611        | 6.10020       
_20200930_GV1-1203_2krpm_XZ-5m-min_2-5H+St... | 507  